# V10 Signed Staged Residual Symbolic Regression Pilot

## Purpose

V10 freezes the locally verified V8 Candidate-5 hierarchy and searches two
signed residual formulas:

`stress = mean + exp(log_scale) * (fixed_shape + geometry_residual + physical_residual)`

The geometry stage learns directional radial, axial and angular boundary
behaviour. The physical stage then learns the remaining local dependence on
fluence rate, temperature and weight-loss rate. Both stages are recoverable.

This is a development-only pilot. It uses 119 training cases, complete 15-case
validation, and one 15-case internal-test report. The 50 final-test cases stay
sealed.


## Why This Version Differs From V9

- The V8 baseline is copied into `shared/` and verified by SHA-256, preventing
  the local Candidate-5 and remote Candidate-11 baselines from being mixed.
- The residual target is computed against deployable V8 predictions, not true
  stress summaries unavailable at deployment.
- Discovery exposure/loss mass is 60/15/20/5% for below-P90, P90-P95,
  P95-P99 and top-P99. V9 exposed top-P99 at 20%, which encouraged broad
  overprediction.
- `HuberLoss(1.0)` reduces domination by extreme residuals while preserving
  tail emphasis through explicit weights.
- `square` is removed because V9 needed negative as well as positive
  corrections. `tanh` and constrained Gaussian localisation remain available.
- Formula selection requires complete-case validation and records explicit
  RMSE, P95/P99, underprediction and hotspot gates. A failed gate remains a
  valid diagnostic result and is not promoted as an engineering formula.

The search design follows the official
[PySR options](https://ai.damtp.cam.ac.uk/pysr/options/),
[tuning guidance](https://ai.damtp.cam.ac.uk/pysr/v2.0.0a2/tuning/) and
[SymbolicRegression loss reference](https://ai.damtp.cam.ac.uk/symbolicregression/dev/losses/)
for batching, weighting, robust loss and constrained operators. Recoverability
uses ordinary PySR checkpoints rather than `TemplateExpressionSpec`, whose
checkpoint serialization has a reported compatibility issue in
[PySR issue 941](https://github.com/MilesCranmer/PySR/issues/941).


## 1. Fresh Kernel and Imports


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "signed_staged_residual_symbolic.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NotebookCT3 package root.")


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from signed_staged_residual_symbolic import (
    SignedStagedPilotConfig,
    output_directory,
    preflight_signed_staged_pilot,
    run_signed_staged_residual_pilot,
)

print("Package root:", PACKAGE_ROOT)
print("Python:", sys.executable)


Package root: /net/scratch/j96317yn/NotebookCT3
Python: /net/scratch/j96317yn/NotebookCT3/.venv/bin/python


## 2. Locked Pilot Configuration


In [2]:
RUN_V10_PILOT = True

CONFIG = SignedStagedPilotConfig(
    iteration=1,
    output_subdir="iteration_1",
    rows_per_training_case=5_000,
    total_niterations_per_stage=500,
    populations=8,
    segment_niterations=100,
    population_size=40,
    ncycles_per_iteration=100,
    batch_size=50_000,
    stage_a_maxsize=32,
    stage_a_maxdepth=10,
    stage_b_maxsize=26,
    stage_b_maxdepth=9,
    julia_threads=8,
    no_activity_timeout_seconds=45 * 60,
    segment_wall_timeout_seconds=3 * 60 * 60,
    watchdog_poll_seconds=60,
    max_attempts_per_segment=3,
    max_candidates_for_full_validation=16,
    force_rebuild_training_cache=False,
)

OUTPUT_DIR = output_directory(PACKAGE_ROOT, CONFIG)
display(pd.DataFrame([CONFIG.__dict__]).T.rename(columns={0: "value"}))
print("Output directory:", OUTPUT_DIR)
print("Segments per stage:", CONFIG.n_segments_per_stage)
print("Population iterations per stage:", CONFIG.population_iterations_per_stage)
print("Two-stage total population iterations:", CONFIG.total_population_iterations)


,value
iteration,1
output_subdir,iteration_1
rows_per_training_case,5000
total_niterations_per_stage,500
populations,8
segment_niterations,100
population_size,40
ncycles_per_iteration,100
batch_size,50000
stage_a_maxsize,32


Output directory: /net/scratch/j96317yn/NotebookCT3/outputs/10_signed_staged_residual_symbolic_pilot/iteration_1
Segments per stage: 5
Population iterations per stage: 4000
Two-stage total population iterations: 8000


## 3. Preflight

Preflight verifies 199 FEM files, the frozen 119/15/15/50 split, all baseline
hashes, 40 total predictors, 28 geometry-stage predictors, 29 physical-stage
predictors and the dedicated worker. It does not read final-test element data.


In [3]:
PREFLIGHT = preflight_signed_staged_pilot(PACKAGE_ROOT, CONFIG)
display(PREFLIGHT["checks"])
display(PREFLIGHT["baseline"]["hash_audit"])
print("Training cases:", len(PREFLIGHT["inputs"]["train_ids"]))
print("Validation cases:", len(PREFLIGHT["inputs"]["validation_ids"]))
print("Internal-test cases:", len(PREFLIGHT["inputs"]["internal_ids"]))
print("Sealed final-test cases:", len(PREFLIGHT["inputs"]["final_ids"]))


,check,value,expected,pass
0,training_cases,119,119,True
1,validation_cases,15,15,True
2,internal_test_cases,15,15,True
3,sealed_final_cases,50,50,True
4,all_v10_features,40,40,True
5,stage_a_features,28,28,True
6,stage_b_features,29,29,True
7,worker_script_exists,True,True,True
8,frozen_baseline_hashes,True,True,True
9,population_iterations_per_stage,4000,4000,True


,file,expected_sha256,actual_sha256,pass
0,case_log_scale_scaling.csv,84d3833edb8e6928212524cc0018e795e382977dd2aece...,84d3833edb8e6928212524cc0018e795e382977dd2aece...,True
1,case_mean_scaling.csv,a903959dba57a356f7ccc912fb53c5ab1c95239cdfcd6a...,a903959dba57a356f7ccc912fb53c5ab1c95239cdfcd6a...,True
2,round_complete.json,2748a84e5f0903d3f3b31d8c8f300c7e1fdcbe5053ecf7...,2748a84e5f0903d3f3b31d8c8f300c7e1fdcbe5053ecf7...,True
3,selected_case_log_scale_formula.csv,c031e909dfd7db9c4dc090a39e46e4bb90c5edfc044e46...,c031e909dfd7db9c4dc090a39e46e4bb90c5edfc044e46...,True
4,selected_case_mean_formula.csv,067bbb98688fec66f8f0780073d3e3fe21e4d4ed9e0e11...,067bbb98688fec66f8f0780073d3e3fe21e4d4ed9e0e11...,True
5,selected_composite_formula.txt,98572e36344e0562312df14458539127946a9c5d7c2095...,98572e36344e0562312df14458539127946a9c5d7c2095...,True
6,selected_formula_split_metrics.csv,100bbdaff1692270d7b2898c541cfb9eeae60a2f8c5db0...,100bbdaff1692270d7b2898c541cfb9eeae60a2f8c5db0...,True
7,selected_shape_formula.csv,42ba93fd8b62f6d564d1d04868752da0994748a5d49367...,42ba93fd8b62f6d564d1d04868752da0994748a5d49367...,True
8,training_cache/shape_scaling.csv,3e2f99f7dce3debd40ec9f8343ea6dde8245eaad2fbf1e...,3e2f99f7dce3debd40ec9f8343ea6dde8245eaad2fbf1e...,True


Training cases: 119
Validation cases: 15
Internal-test cases: 15
Sealed final-test cases: 50


## 4. Run or Resume Both Stages

Rerunning this cell reuses the sample cache and every completed segment. Stage
B begins only after Stage A has been fully validated and selected. Search logs,
watchdog states and checkpoint snapshots are stored separately under
`stages/stage_geometry/` and `stages/stage_physical/`.


In [4]:
RESULT = None
if RUN_V10_PILOT:
    RESULT = run_signed_staged_residual_pilot(PACKAGE_ROOT, CONFIG)
    display(pd.DataFrame([RESULT]))
else:
    print("V10 pilot skipped because RUN_V10_PILOT=False.")


[1/119] V10 signed-residual sample: case_01


[2/119] V10 signed-residual sample: case_10


[3/119] V10 signed-residual sample: case_11


[4/119] V10 signed-residual sample: case_12


[5/119] V10 signed-residual sample: case_13


...

[12/15] internal_test complete-case V10 physical: case_159


[13/15] internal_test complete-case V10 physical: case_196


[14/15] internal_test complete-case V10 physical: case_197


[15/15] internal_test complete-case V10 physical: case_199


,status,iteration,prototype_only,method,baseline_id,train_cases,validation_cases,internal_test_cases,final_test_cases_read,training_rows,planned_population_iterations,geometry_completed_segments,physical_completed_segments,selected_geometry_candidate,selected_physical_candidate,pilot_promotion_status,elapsed_seconds,output_directory
0,complete,1,True,frozen_v8_plus_signed_geometry_then_physical_r...,v8_iteration1_local_candidate5,119,15,15,0,595000,8000,5,5,3,0,diagnostic_only_does_not_pass_all_v10_pilot_gates,6881.406506,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...


## 5. Saved Evidence


In [5]:
artifacts = {
    "completion": OUTPUT_DIR / "pilot_complete.json",
    "formula": OUTPUT_DIR / "selected_composite_formula.txt",
    "split_metrics": OUTPUT_DIR / "selected_models_split_metrics.csv",
    "tail_metrics": OUTPUT_DIR / "selected_models_tail_adaptation.csv",
    "geometry_candidates": OUTPUT_DIR / "stages" / "stage_geometry" / "candidate_validation_metrics.csv",
    "physical_candidates": OUTPUT_DIR / "stages" / "stage_physical" / "candidate_validation_metrics.csv",
    "sampling_audit": OUTPUT_DIR / "training_cache" / "training_sample_audit.csv",
}
display(pd.DataFrame([
    {"artifact": name, "exists": path.exists(), "path": str(path)}
    for name, path in artifacts.items()
]))

if artifacts["completion"].exists():
    print(artifacts["formula"].read_text(encoding="utf-8"))
    display(pd.read_csv(artifacts["split_metrics"]))
    display(pd.read_csv(artifacts["tail_metrics"]))
    display(
        pd.read_csv(artifacts["physical_candidates"])
        .sort_values("engineering_selection_score")
        .head(16)
    )


,artifact,exists,path
0,completion,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...
1,formula,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...
2,split_metrics,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...
3,tail_metrics,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...
4,geometry_candidates,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...
5,physical_candidates,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...
6,sampling_audit,True,/net/scratch/j96317yn/NotebookCT3/outputs/10_s...


CT3 V10 signed staged symbolic stress formula

Frozen baseline: v8_iteration1_local_candidate5

1. Case mean
mu = -0.0116301024532845*temperature_mean + 0.0470427355318997*temperature_p95 - 3.16087946558646*weight_loss_rate_mean + 0.103449215368151*z_max - 0.870195660324862*Abs(4.60760803334225*fluence_rate_p95 - 20.3572633494299) - 100.195330145459

2. Positive case scale
scale = exp(-1.47091131932291*rho_mean - 1011.416708227*theta_sin_std - 1.1690770556861*weight_loss_rate_mean + 0.904578545888617*z_mean + 9.23869712445276)

3. Frozen V8 global shape
fixed_shape = 1.10317833861391*(0.55673575*(0.0509611749551904*rho - 9.32241465638866)*(0.0509611749551904*rho - 8.66867535638866) - 31.9731949759449*(theta_cos - 0.922666019258146)**2)*(Abs(0.0509611749551904*rho - 8.59575422638866) - 2.4201684) + 0.285739561816426

4. Signed geometry/boundary residual
geometry_residual = 0.0005298607274781943*z + 0.17657582192765694*(6.3559467282562*nearest_axial_boundary_fraction_proxy - 1.5057042488

,iteration,split,model,n_cases,n_elements_evaluated,micro_mae,micro_rmse,micro_r2,macro_mae,macro_rmse,...,mean_top5_actual_rmse,mean_top5_actual_bias,mean_p95_relative_error,mean_p95_underprediction_fraction,mean_p99_relative_error,mean_p99_underprediction_fraction,mean_top5pct_hotspot_overlap,mean_top1pct_hotspot_overlap,mean_top1_recall_in_predicted_top5,max_prediction_abs_max_ratio
0,1,internal_test,frozen_v8_candidate5,15,6005400,1.847777,2.471889,0.281895,1.847777,2.451950,...,5.110821,-3.972092,0.150652,0.150652,0.253503,0.253503,0.319945,0.447236,0.609590,1.166592
1,1,internal_test,v10_geometry_residual_only,15,6005400,1.463231,2.102364,0.480548,1.463231,2.071896,...,4.059158,-2.404154,0.031550,0.007782,0.122509,0.120754,0.475016,0.416084,0.799500,1.450343
2,1,internal_test,v10_signed_staged_symbolic,15,6005400,1.334879,1.908915,0.571744,1.334879,1.893249,...,3.724246,-2.177258,0.023138,0.007712,0.147758,0.141902,0.525917,0.509374,0.823726,1.444977
3,1,validation,frozen_v8_candidate5,15,6005400,1.768228,2.403750,0.358755,1.768228,2.347327,...,4.630008,-3.574538,0.128592,0.118436,0.214801,0.214801,0.307007,0.488994,0.600200,0.838627
4,1,validation,v10_geometry_residual_only,15,6005400,1.426936,2.064708,0.526889,1.426936,2.025636,...,3.649761,-2.038539,0.065780,0.002700,0.108851,0.088841,0.469764,0.422894,0.804829,1.027313
5,1,validation,v10_signed_staged_symbolic,15,6005400,1.331691,1.919506,0.591093,1.331691,1.888373,...,3.404932,-1.927182,0.076856,0.010752,0.126061,0.109526,0.508013,0.518198,0.826607,1.033887


,split,model,case_p95_spearman,case_p95_std_ratio,case_p95_mean_bias,case_p99_spearman,case_p99_std_ratio,case_p99_mean_bias
0,internal_test,frozen_v8_candidate5,0.967857,0.880297,-1.343237,0.767857,0.861680,-3.594022
1,internal_test,v10_geometry_residual_only,0.932143,0.999958,0.136334,0.767857,0.992673,-1.693533
2,internal_test,v10_signed_staged_symbolic,0.960714,1.037905,0.068611,0.767857,1.115481,-1.930933
3,validation,frozen_v8_candidate5,0.842857,0.722500,-1.093643,0.825000,0.661624,-3.088853
4,validation,v10_geometry_residual_only,0.928571,0.861929,0.329878,0.825000,0.772452,-1.199188
5,validation,v10_signed_staged_symbolic,0.792857,0.836651,0.260390,0.825000,0.735570,-1.538484


,stage,run_id,candidate_index,complexity,loss,score,equation,formula_scaled_sympy,formula_original_variables,feature_support_json,...,gate_p95_relative_error,gate_p99_relative_error,gate_p95_underprediction,gate_p99_underprediction,gate_top1_hotspot_overlap,gate_top1_recall,macro_rmse_improvement_vs_stage_a_fraction,gate_not_worse_than_stage_a_rmse,all_final_promotion_gates_pass,selected_candidate
15,v10_physical_residual,v10_i1_physical_signed_residual_pilot,0,26,0.328641,0.001423,(((((theta_fraction_proxy_scaled + abs(signed_...,((fluence_rate_mean_scaled + nearest_angular_b...,0.687707530043104*(3.752650699651312*nearest_a...,"[""fluence_rate_mean_scaled"", ""nearest_angular_...",...,True,True,True,False,False,True,0.067763,True,False,True
14,v10_physical_residual,v10_i1_physical_signed_residual_pilot,2,23,0.329808,0.001745,(((theta_fraction_proxy_scaled + abs(signed_ra...,(boundary_corner_proximity_proxy_scaled + near...,0.687707530043104*(43.6680362093901*boundary_c...,"[""boundary_corner_proximity_proxy_scaled"", ""fl...",...,True,True,True,False,False,True,0.070771,True,False,False
11,v10_physical_residual,v10_i1_physical_signed_residual_pilot,5,18,0.334525,0.010226,(nearest_radial_boundary_fraction_proxy_scaled...,(nearest_angular_boundary_fraction_proxy_scale...,0.10507361105129148*(3.096429194021555*nearest...,"[""fluence_rate_mean_scaled"", ""nearest_angular_...",...,True,True,True,True,False,True,0.029169,True,False,False
13,v10_physical_residual,v10_i1_physical_signed_residual_pilot,3,21,0.330961,0.003327,((nearest_radial_boundary_fraction_proxy_scale...,(boundary_corner_proximity_proxy_scaled + near...,-0.082214749397234079*(43.6680362093901*bounda...,"[""boundary_corner_proximity_proxy_scaled"", ""fl...",...,True,True,True,False,False,True,0.051963,True,False,False
7,v10_physical_residual,v10_i1_physical_signed_residual_pilot,9,12,0.342738,0.003598,(nearest_radial_boundary_fraction_proxy_scaled...,nearest_radial_boundary_fraction_proxy_scaled*...,0.14103595535763298*(7.28799975835273*nearest_...,"[""fluence_rate_mean_scaled"", ""nearest_axial_bo...",...,True,True,True,True,False,True,-0.001043,False,False,False
8,v10_physical_residual,v10_i1_physical_signed_residual_pilot,8,14,0.341629,0.001621,nearest_radial_boundary_fraction_proxy_scaled ...,nearest_radial_boundary_fraction_proxy_scaled*...,0.10926064548169548*(7.28799975835273*nearest_...,"[""fluence_rate_mean_scaled"", ""nearest_angular_...",...,True,True,True,True,False,True,0.014203,True,False,False
12,v10_physical_residual,v10_i1_physical_signed_residual_pilot,4,19,0.333171,0.004056,(nearest_radial_boundary_fraction_proxy_scaled...,(boundary_corner_proximity_proxy_scaled + near...,-0.0652259935378015*(43.6680362093901*boundary...,"[""boundary_corner_proximity_proxy_scaled"", ""fl...",...,True,True,True,False,False,True,0.036647,True,False,False
6,v10_physical_residual,v10_i1_physical_signed_residual_pilot,10,11,0.343974,0.008332,(nearest_angular_boundary_fraction_proxy_scale...,(nearest_angular_boundary_fraction_proxy_scale...,0.075347054887131483*(7.94611837518369*nearest...,"[""fluence_rate_mean_scaled"", ""nearest_angular_...",...,True,True,True,False,False,True,0.066977,True,False,False
5,v10_physical_residual,v10_i1_physical_signed_residual_pilot,11,10,0.346851,0.006445,nearest_radial_boundary_fraction_proxy_scaled ...,nearest_radial_boundary_fraction_proxy_scaled*...,-0.1309463157713888*(7.28799975835273*nearest_...,"[""fluence_rate_mean_scaled"", ""nearest_radial_b...",...,True,True,True,True,False,True,-0.013449,False,False,False
10,v10_physical_residual,v10_i1_physical_signed_residual_pilot,6,17,0.337963,0.002326,((nearest_angular_boundary_fraction_proxy_scal...,(fluence_rate_mean_scaled + nearest_angular_bo...,-0.099038508810301796*(4.556415851869832*neare...,"[""fluence_rate_mean_scaled"", ""nearest_angular_...",...,True,True,True,True,False,True,0.013489,True,False,False


## Interpretation Boundary

V10 tests whether a readable two-stage correction can beat the frozen V8
baseline without sacrificing P95/P99 or hotspot localisation. Passing every
validation gate supports a larger formal search. Failure means that this
symbolic search space has not justified promotion; it does not authorize use
of the diagnostic formula as an engineering replacement for FEM.

This remains a stress surrogate, not a lifetime model. Lifetime conversion
still requires a professor-confirmed strength, damage or failure relationship.
